# Child Mortality Analysis
### Predicting Under-5 Mortality Across Countries Using World Development Indicators

Group 5 of "Einführung in maschinelles Lernen" - A Data Science Project.
- Ayoub Taychi
- Hajar Lasri
- Yama Saputra

## 1.1 Domain Problem
### Research Questions
#### 1. Can ML regression models predict under-5 child mortality rates across countries using health, education, income, sanitation, and governance indicators from the WDI?
Justification: Under-5 mortality is a socially relevant and policy-important development outcome that is strontly linked to health, sanitation, income, and demographic conditions.
#### 2. Can ML classifiers predict whether a country-year will fall into a high under-5 mortality risk group in the following year?
Justification: High risk is defined as the top 25% next-year under-5 mortality values, which creates a simple and interpretable binary target.

Technical Environment:
Python v3.13.13
Importing data project libraries, pandas and scikit learn.

In [1]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

Current running version environments for the libraries (freezed with requirements.txt for shareability):

In [2]:
print(f"pandas: {pd.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"sklearn: {sklearn.__version__}")

pandas: 2.3.1
matplotlib: 3.10.1
sklearn: 1.6.1


install with

`
pip install -r requirements.txt
`

## 1.2 Data Collection
Dataset utilized is from the World Bank WDI (https://data360.worldbank.org/en/dataset/WB_WDI). Observed are over 295 thousand rows, 107 columns, 1516 indicators from 217 countries.

- Initial dataset was too large containing too many irrelevant features.
- Dataset was reshaped with the help of chatbot Claude Opus 4.7 to reduce dimensionality and restructured from a messy wide to a tidy long format.
- Each row is one country-year observation, each feature is its own column. The convenient shape for predictive models.




In [3]:
# Load the dataset
df = pd.read_csv('WDI_under5_mortality_features.csv')
print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")

Dataset loaded successfully!
Dataset shape: (14181, 27)


In [4]:
# Display first few rows
print("=" * 80)
print("FIRST 5 ROWS OF THE DATASET")
print("=" * 80)
df.head()

FIRST 5 ROWS OF THE DATASET


,country_code,country_name,year,under5_mortality_per_1000,gdp_per_capita_ppp_current,gdp_per_capita_ppp_constant_2021,gini_index,health_expenditure_pct_gdp,physicians_per_1000,primary_completion_rate_pct,...,measles_immunization_pct,rule_of_law_estimate,rule_of_law_percentile,govt_effectiveness_estimate,govt_effectiveness_percentile,fertility_rate_births_per_woman,urban_population_pct,life_expectancy_years,neonatal_mortality_per_1000,population_total
0,AFG,Afghanistan,1960,353.2,NaN,NaN,NaN,NaN,0.035,NaN,...,NaN,NaN,NaN,NaN,NaN,7.282,8.122551,32.799,NaN,9035043.0
1,AFG,Afghanistan,1961,347.6,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.284,8.389595,33.291,NaN,9214083.0
2,AFG,Afghanistan,1962,342.3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.292,8.662664,33.757,NaN,9404406.0
3,AFG,Afghanistan,1963,336.8,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.302,8.942709,34.201,NaN,9604487.0
4,AFG,Afghanistan,1964,331.7,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.304,9.231134,34.673,NaN,9814318.0


In [5]:
# Dataset overview and info
print("\n" + "=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nBasic statistics:\n{df.describe()}")


DATASET OVERVIEW
Total rows: 14181
Total columns: 27

Data types:
country_code                         object
country_name                         object
year                                  int64
under5_mortality_per_1000           float64
gdp_per_capita_ppp_current          float64
gdp_per_capita_ppp_constant_2021    float64
gini_index                          float64
health_expenditure_pct_gdp          float64
physicians_per_1000                 float64
primary_completion_rate_pct         float64
adult_literacy_rate_pct             float64
basic_drinking_water_pct            float64
safely_managed_water_pct            float64
basic_sanitation_pct                float64
safely_managed_sanitation_pct       float64
electricity_access_pct              float64
dpt_immunization_pct                float64
measles_immunization_pct            float64
rule_of_law_estimate                float64
rule_of_law_percentile              float64
govt_effectiveness_estimate         float64
govt_effe

In [6]:
# Missing values analysis
print("\n" + "=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Column': missing_count.index, 'Missing Count': missing_count.values, 'Missing %': missing_percentage.values})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(f"Total missing values in dataset: {df.isnull().sum().sum()}")
print(f"\nColumns with missing values:\n{missing_df}")


MISSING VALUES ANALYSIS
Total missing values in dataset: 170207

Columns with missing values:
                              Column  Missing Count  Missing %
10           adult_literacy_rate_pct          13035  91.918765
6                         gini_index          11779  83.061843
14     safely_managed_sanitation_pct          10612  74.832522
12          safely_managed_water_pct          10465  73.795924
7         health_expenditure_pct_gdp           9595  67.660955
9        primary_completion_rate_pct           9360  66.003808
20       govt_effectiveness_estimate           9217  64.995416
21     govt_effectiveness_percentile           9217  64.995416
18              rule_of_law_estimate           9103  64.191524
19            rule_of_law_percentile           9103  64.191524
13              basic_sanitation_pct           8990  63.394683
11          basic_drinking_water_pct           8947  63.091460
8                physicians_per_1000           8818  62.181793
15            electrici

In [7]:
# Countries analysis
print("\n" + "=" * 80)
print("COUNTRIES ANALYSIS")
print("=" * 80)
# Assuming there's a 'Country' or similar column - adjust column name if needed
country_col = [col for col in df.columns if 'country' in col.lower()][0] if any('country' in col.lower() for col in df.columns) else None
if country_col:
    countries_sorted = df[country_col].drop_duplicates().sort_values()
    print(f"Total number of unique countries: {df[country_col].nunique()}")
    print(f"\nCountries ordered alphabetically:\n{countries_sorted.to_list()}")
else:
    print("No country column found. Available columns:", df.columns.tolist())


COUNTRIES ANALYSIS
Total number of unique countries: 218

Countries ordered alphabetically:
['ABW', 'AFG', 'AGO', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHI', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CPV', 'CRI', 'CUB', 'CUW', 'CYM', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FRA', 'FRO', 'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD', 'GRL', 'GTM', 'GUM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IMN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LIE', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAC', 'MAF', 'MAR', 'MCO', 'MDA', 'MDG', 'MDV', 'MEX', 'MHL', 'MKD', 'MLI', 'ML

In [8]:
# Available features
print("\n" + "=" * 80)
print("AVAILABLE FEATURES")
print("=" * 80)
print(f"Total number of features: {len(df.columns)}")
print(f"\nAll features:\n")
for idx, col in enumerate(df.columns, 1):
    print(f"{idx}. {col}")


AVAILABLE FEATURES
Total number of features: 27

All features:

1. country_code
2. country_name
3. year
4. under5_mortality_per_1000
5. gdp_per_capita_ppp_current
6. gdp_per_capita_ppp_constant_2021
7. gini_index
8. health_expenditure_pct_gdp
9. physicians_per_1000
10. primary_completion_rate_pct
11. adult_literacy_rate_pct
12. basic_drinking_water_pct
13. safely_managed_water_pct
14. basic_sanitation_pct
15. safely_managed_sanitation_pct
16. electricity_access_pct
17. dpt_immunization_pct
18. measles_immunization_pct
19. rule_of_law_estimate
20. rule_of_law_percentile
21. govt_effectiveness_estimate
22. govt_effectiveness_percentile
23. fertility_rate_births_per_woman
24. urban_population_pct
25. life_expectancy_years
26. neonatal_mortality_per_1000
27. population_total
